In [5]:

import numpy as np
import random

# City coordinates (x, y)
cities = np.array([
    [0, 0],
    [2, 6],
    [5, 3],
    [7, 8],
    [8, 2],
    [1, 4]
])

# Calculate Euclidean distances
n = len(cities)
distance = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        distance[i][j] = np.linalg.norm(
            cities[i] - cities[j]
        )

# ACO parameters
num_ants = 10
num_iterations = 20
alpha = 1.0
beta = 5.0
rho = 0.5
Q = 100

# Initialize pheromone matrix
pheromone = np.ones((n, n))

best_route = None
best_distance = float("inf")

# Ant Colony Optimization
for iteration in range(num_iterations):

    routes = []
    route_distances = []

    for ant in range(num_ants):

        # Select starting city
        start = random.randint(0, n - 1)
        route = [start]

        unvisited = set(range(n))
        unvisited.remove(start)

        # Construct route
        while unvisited:
            current = route[-1]
            candidates = list(unvisited)

            weights = []

            for city in candidates:
                tau = pheromone[current][city] ** alpha
                eta = (1.0 / (
                    distance[current][city] + 1e-10
                )) ** beta

                weights.append(tau * eta)

            probabilities = np.array(weights, dtype=float)
            probabilities /= probabilities.sum()

            next_city = random.choices(
                candidates,
                weights=probabilities,
                k=1
            )[0]

            route.append(next_city)
            unvisited.remove(next_city)

        # Return to starting city
        route.append(route[0])

        # Calculate total route distance
        total_distance = sum(
            distance[route[i]][route[i + 1]]
            for i in range(n)
        )

        routes.append(route)
        route_distances.append(total_distance)

        # Store the best route
        if total_distance < best_distance:
            best_distance = total_distance
            best_route = route.copy()

    # Evaporate pheromones
    pheromone *= (1 - rho)

    # Update pheromones based on route quality
    for route, length in zip(routes, route_distances):
        deposit = Q / length

        for i in range(n):
            a = route[i]
            b = route[i + 1]

            pheromone[a][b] += deposit
            pheromone[b][a] += deposit

print("Best Route:")
print(" -> ".join(map(str, best_route)))

print(f"Minimum Distance: {best_distance:.2f} units")


Best Route:
2 -> 4 -> 3 -> 1 -> 5 -> 0 -> 2
Minimum Distance: 26.82 units
